# Heart Disease Severity — Final Model Program
### LightGBM · XGBoost · CatBoost · Logistic Regression · Ordinal LightGBM · two Stacking Ensembles

Companion to `Heart_Disease_Severity_ML_Pipeline_fixed.ipynb` (SVM, Random Forest, MLP). This notebook
does **not** modify it. It uses the **same** cleaning, features, 80/20 split (seed 42), preprocessing,
SMOTE-inside-the-pipeline design, severity-weighted-F1 tuning and 5×3 nested CV, so all results are
comparable with your existing three models.

**Models run here**

| Model | Why it is in the paper |
|---|---|
| LightGBM | Your Azure AutoML winner (`LightGBM + MaxAbsScaler`), used as the search starting point |
| XGBoost | Strongest single model in the comparator paper (65.8%) |
| **CatBoost** *(recommended addition)* | Ordered boosting is very stable on small tabular data (~740 rows); the natural third boosting baseline |
| **Logistic Regression** *(recommended addition)* | Linear baseline. Shows how much non-linear models actually add, and reviewers expect one |
| **Ordinal LightGBM** *(recommended addition)* | `num` is an *ordered* severity scale (0<1<2<3<4). Regress on it, then learn cut-points that maximise severity-F1. Uses the ordering that plain classifiers ignore |
| **Stack-A** — LightGBM + XGBoost + CatBoost → LR meta | Homogeneous "boosting" stack |
| **Stack-B** — LightGBM + XGBoost + RF + SVM + MLP → LR meta | Heterogeneous stack (needs SVM/RF/MLP as base learners) |

**Metrics reported everywhere:** accuracy, macro-F1, severity-weighted F1 (your scorer), and two
ordinal-aware metrics that suit a severity target — **quadratic weighted kappa** and **adjacent
accuracy** (prediction within ±1 severity level) — plus recall on the two most severe classes.


## Step 1: Imports & configuration

In [ ]:
import os, sys, subprocess, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import stats
warnings.filterwarnings("ignore")

def _ensure(pkg, mod=None):
    try:
        __import__(mod or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
_ensure("lightgbm"); _ensure("xgboost"); _ensure("catboost"); _ensure("imbalanced-learn", "imblearn")

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MaxAbsScaler, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_val_predict
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, f1_score, recall_score, cohen_kappa_score, make_scorer,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ── Settings (identical to the original notebook unless noted) ────────────────
SEED       = 42
TEST_SIZE  = 0.20
N_OUTER    = 5          # outer CV folds
N_INNER    = 3          # inner CV folds (hyper-parameter search)
N_REPEATS  = 1          # >1 repeats the outer CV with new shuffles. Use 3 for the paper so the
                        # Wilcoxon tests in Step 9 have enough folds (5x3 = 15 paired scores)
N_JOBS     = os.cpu_count() or 1

QUICK_MODE                = False   # True = 1 grid point per model + 3 outer folds (smoke test only)
RERUN_BASELINES           = False   # True = also re-run SVM / RF / MLP so one table has every model
INCLUDE_CATBOOST          = True
INCLUDE_ORDINAL           = True
USE_DATASET_SITE_FEATURE  = False   # False = clinical features only (recommended, see notes at the end)

if QUICK_MODE:
    N_OUTER = 3

TARGET       = "num"
CLASS_NAMES  = ["No Disease", "Mild", "Moderate", "Severe", "Very Severe"]
SEV_WEIGHTS  = {0: 1.0, 1: 1.5, 2: 2.0, 3: 3.0, 4: 4.0}
np.random.seed(SEED)
print(f"SEED={SEED}  outer={N_OUTER} x inner={N_INNER}  repeats={N_REPEATS}  workers={N_JOBS}  quick={QUICK_MODE}")


## Step 2: Load, clean, engineer features, split

Same as the original notebook, with **one fix**: `cp_risk` now compares against the string categories that are actually in the CSV (the original compared against `[0, 3]`, which never matches, so that feature was constant).

In [ ]:
df = pd.read_csv("heart_disease_uci.csv")
df.columns = df.columns.str.strip().str.lower()
df.drop_duplicates(inplace=True)

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalch", "exang", "oldpeak", "slope", "ca", "thal"]
NUMERIC_COLS  = ["age", "trestbps", "chol", "thalch", "oldpeak"]
CATEGORY_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
if USE_DATASET_SITE_FEATURE:
    FEATURES = FEATURES + ["dataset"]
    CATEGORY_COLS = CATEGORY_COLS + ["dataset"]

for col, (lo, hi) in {"chol": (100, 600), "trestbps": (60, 220), "thalch": (60, 220),
                      "oldpeak": (0, 10), "age": (20, 100)}.items():
    df[col] = df[col].clip(lo, hi)

df["age_group"]    = pd.cut(df["age"], bins=[20, 40, 55, 65, 100], labels=[0, 1, 2, 3]).astype("float")
df["cp_risk"]      = np.where(df["cp"].isna(), np.nan, df["cp"].isin(["typical angina", "asymptomatic"]).astype(float))  # FIXED
df["high_hr"]      = np.where(df["thalch"].isna(), np.nan, (df["thalch"] > 150).astype(float))
df["hypertension"] = np.where(df["trestbps"].isna(), np.nan, (df["trestbps"] > 140).astype(float))
df["oldpeak_cat"]  = pd.cut(df["oldpeak"], bins=[-0.1, 0.5, 2.0, 10.0], labels=[0, 1, 2]).astype("float")
ENGINEERED   = ["age_group", "cp_risk", "high_hr", "hypertension", "oldpeak_cat"]
ALL_FEATURES = FEATURES + ENGINEERED
assert df["cp_risk"].nunique() == 2, "cp_risk should be a real 0/1 feature"

X = df[ALL_FEATURES].copy()
y = df[TARGET].values.astype(int)
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
print(f"train={len(X_train)}  holdout={len(X_holdout)}  features={len(ALL_FEATURES)}")
print("train class counts  :", dict(zip(*np.unique(y_train, return_counts=True))))
print("holdout class counts:", dict(zip(*np.unique(y_holdout, return_counts=True))))


## Step 3: Preprocessing, scorer and metrics

In [ ]:
NUMERIC_FEATURES     = NUMERIC_COLS + ENGINEERED
CATEGORICAL_FEATURES = CATEGORY_COLS

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NUMERIC_FEATURES),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
         CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

def severity_weighted_f1(y_true, y_pred):
    per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
    w = np.array([SEV_WEIGHTS[i] for i in range(5)])
    return float(np.dot(per_class, w / w.sum()))
severity_scorer = make_scorer(severity_weighted_f1)

def all_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rec = recall_score(y_true, y_pred, labels=[0, 1, 2, 3, 4], average=None, zero_division=0)
    return {
        "accuracy":     accuracy_score(y_true, y_pred),
        "macro_f1":     f1_score(y_true, y_pred, average="macro", zero_division=0),
        "severity_f1":  severity_weighted_f1(y_true, y_pred),
        "qwk":          cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "adjacent_acc": float(np.mean(np.abs(y_true - y_pred) <= 1)),
        "recall_C3":    rec[3],
        "recall_C4":    rec[4],
    }
METRICS = ["accuracy", "macro_f1", "severity_f1", "qwk", "adjacent_acc", "recall_C3", "recall_C4"]
print("preprocessing, scorer and metrics ready")


## Step 4: Model definitions

Every model is `Preprocess → SMOTE → Scaler → Classifier` (SMOTE only ever sees training folds), exactly
like SVM/RF/MLP in the original. LightGBM's starting point is your **Azure AutoML** configuration
(with `min_data_in_leaf` converted from a fraction to a sample count).

* Individual models are **tuned** inside nested CV (small grid around a sensible/Azure point).
* Base learners inside the two **stacks** use **fixed** hyper-parameters, so no hyper-parameter selection ever sees the outer test folds.
* **Ordinal LightGBM** has no SMOTE step — class imbalance is handled by choosing the cut-points to maximise severity-F1.

In [ ]:
MIN_LEAF = max(1, round(0.027593448275862072 * len(X_train)))    # Azure's min_data_in_leaf fraction -> samples

LGBM_PARAMS = dict(boosting_type="gbdt", colsample_bytree=0.8911111111111111, learning_rate=0.07368684210526316,
                   max_bin=380, max_depth=2, min_child_weight=5, min_child_samples=MIN_LEAF,
                   min_split_gain=0.631578947368421, n_estimators=800, num_leaves=86,
                   reg_alpha=0.7894736842105263, reg_lambda=0.3684210526315789,
                   subsample=0.8910526315789474, subsample_freq=1,
                   random_state=SEED, n_jobs=1, verbose=-1)

def smote_pipe(clf, scaler=None):
    return ImbPipeline([("preprocess", preprocessor), ("smote", SMOTE(random_state=SEED)),
                        ("scaler", scaler if scaler is not None else StandardScaler()), ("clf", clf)])

def xgb_clf(**kw):
    p = dict(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", subsample=0.8,
             colsample_bytree=0.8, random_state=SEED, n_jobs=1)
    p.update(kw); return XGBClassifier(**p)

def cat_clf(**kw):
    p = dict(iterations=300, loss_function="MultiClass", random_seed=SEED, thread_count=1, verbose=0,
             allow_writing_files=False)
    p.update(kw); return CatBoostClassifier(**p)

# ── Ordinal model: regression on the severity scale + learned cut-points ──────
class OrdinalThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, regressor=None, optimize_thresholds=True, n_splits=3, random_state=42):
        self.regressor = regressor
        self.optimize_thresholds = optimize_thresholds
        self.n_splits = n_splits
        self.random_state = random_state

    def _optimise(self, oof, y, th):
        th = th.copy()
        best = severity_weighted_f1(y, np.digitize(oof, th))
        for _ in range(3):                                   # coordinate descent over the 4 cut-points
            for k in range(4):
                lo = th[k - 1] + 0.02 if k > 0 else -np.inf
                hi = th[k + 1] - 0.02 if k < 3 else np.inf
                for cand in np.arange(k + 0.5 - 0.6, k + 0.5 + 0.6, 0.05):
                    if not (lo < cand < hi):
                        continue
                    trial = th.copy(); trial[k] = cand
                    s = severity_weighted_f1(y, np.digitize(oof, trial))
                    if s > best + 1e-12:
                        best, th = s, trial
        return th

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        th = np.array([0.5, 1.5, 2.5, 3.5])
        if self.optimize_thresholds:
            cv = StratifiedKFold(self.n_splits, shuffle=True, random_state=self.random_state)
            oof = cross_val_predict(clone(self.regressor), X, y, cv=cv)
            th = self._optimise(oof, y, th)
        self.thresholds_ = th
        self.regressor_ = clone(self.regressor).fit(X, y)
        return self

    def predict(self, X):
        return np.digitize(self.regressor_.predict(X), self.thresholds_)

def ordinal_pipe():
    reg = LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=3, subsample=0.8,
                        subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, min_child_samples=MIN_LEAF,
                        random_state=SEED, n_jobs=1, verbose=-1)
    return Pipeline([("preprocess", preprocessor), ("scaler", StandardScaler()),
                     ("clf", OrdinalThresholdClassifier(regressor=reg, random_state=SEED))])

# ── Baselines (exact copies of the original notebook's definitions) ───────────
def svm_pipe():
    return smote_pipe(SVC(probability=True, class_weight="balanced", random_state=SEED, max_iter=2000, cache_size=400))
def rf_pipe():
    return smote_pipe(RandomForestClassifier(max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=1))
def mlp_pipe():
    return smote_pipe(MLPClassifier(max_iter=500, random_state=SEED, early_stopping=True))

# ── Registry: (pipeline, hyper-parameter grid) for the individually-tuned models ─
MODEL_REGISTRY = {
    "LightGBM": (smote_pipe(LGBMClassifier(**LGBM_PARAMS), MaxAbsScaler()),
                 {"clf__n_estimators": [300, 800], "clf__learning_rate": [0.05, 0.07368684210526316], "clf__max_depth": [2, 4]}),
    "XGBoost":  (smote_pipe(xgb_clf()),
                 {"clf__n_estimators": [300, 600], "clf__max_depth": [3, 5], "clf__learning_rate": [0.03, 0.07]}),
    "Logistic Regression": (smote_pipe(LogisticRegression(max_iter=3000, random_state=SEED)),
                 {"clf__C": [0.1, 1.0, 10.0]}),
}
if INCLUDE_CATBOOST:
    MODEL_REGISTRY["CatBoost"] = (smote_pipe(cat_clf()), {"clf__depth": [4, 6], "clf__learning_rate": [0.03, 0.07]})
if INCLUDE_ORDINAL:
    MODEL_REGISTRY["Ordinal LightGBM"] = (ordinal_pipe(),
                 {"clf__regressor__max_depth": [2, 4], "clf__regressor__n_estimators": [200, 400]})
if RERUN_BASELINES:
    MODEL_REGISTRY["SVM"]           = (svm_pipe(), {"clf__C": [0.1, 1.0], "clf__kernel": ["rbf"], "clf__gamma": ["scale"]})
    MODEL_REGISTRY["Random Forest"] = (rf_pipe(),  {"clf__n_estimators": [100, 200], "clf__max_depth": [10, None], "clf__min_samples_split": [2, 5]})
    MODEL_REGISTRY["MLP"]           = (mlp_pipe(), {"clf__hidden_layer_sizes": [(64, 32), (128, 64)], "clf__alpha": [0.0001],
                                                     "clf__activation": ["relu"], "clf__learning_rate_init": [0.001, 0.0005]})
if QUICK_MODE:
    MODEL_REGISTRY = {n: (p, {k: v[:1] for k, v in g.items()}) for n, (p, g) in MODEL_REGISTRY.items()}

# ── Stacks: fixed-hyper-parameter base learners, Logistic-Regression meta-learner ─
def fixed_lgbm(): return smote_pipe(LGBMClassifier(**LGBM_PARAMS), MaxAbsScaler())
def fixed_xgb():  return smote_pipe(xgb_clf(n_estimators=300, max_depth=4, learning_rate=0.05))
def fixed_cat():  return smote_pipe(cat_clf(depth=6, learning_rate=0.05))

STACK_A_NAME = "Stack-A (LGBM+XGB+CatBoost)" if INCLUDE_CATBOOST else "Stack-A (LGBM+XGB)"
STACK_B_NAME = "Stack-B (LGBM+XGB+RF+SVM+MLP)"

def make_stack(name):
    if name == STACK_A_NAME:
        base = [("lgbm", fixed_lgbm()), ("xgb", fixed_xgb())] + ([("cat", fixed_cat())] if INCLUDE_CATBOOST else [])
    else:
        base = [("lgbm", fixed_lgbm()), ("xgb", fixed_xgb()), ("rf", rf_pipe()), ("svm", svm_pipe()), ("mlp", mlp_pipe())]
    return StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=2000, random_state=SEED),
                              cv=StratifiedKFold(N_INNER, shuffle=True, random_state=SEED),
                              stack_method="predict_proba", n_jobs=1, passthrough=False)
STACK_NAMES = [STACK_A_NAME, STACK_B_NAME]

print("Individually tuned models :", list(MODEL_REGISTRY))
print("Stacking ensembles        :", STACK_NAMES)


## Step 5: Nested cross-validation (individual models)

Same protocol as the original: outer folds give the honest estimate, inner `GridSearchCV` (scored on severity-F1) picks hyper-parameters. Fold 0 of repeat 0 uses exactly the same split as the original notebook.

In [ ]:
outer_splits = []
for r in range(N_REPEATS):
    skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED + r)
    for f, (tr, te) in enumerate(skf.split(X_train, y_train)):
        outer_splits.append((f"r{r}f{f}", tr, te))
inner_cv = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=SEED)

def _nested_fold(name, pipe, grid, split_id, tr, te):
    gs = GridSearchCV(pipe, grid, cv=inner_cv, scoring=severity_scorer, n_jobs=1, refit=True, error_score="raise")
    gs.fit(X_train.iloc[tr], y_train[tr])
    return {"model": name, "split": split_id, **all_metrics(y_train[te], gs.predict(X_train.iloc[te]))}

t0 = time.time()
tasks = [(n, p, g, sid, tr, te) for n, (p, g) in MODEL_REGISTRY.items() for (sid, tr, te) in outer_splits]
print(f"{len(tasks)} (model x outer-fold) tasks on {N_JOBS} worker(s) ...")
rows = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(delayed(_nested_fold)(*t) for t in tasks)
print(f"nested CV finished in {time.time()-t0:.0f}s")


## Step 6: Stacking ensembles on the same outer folds

Base learners use fixed hyper-parameters, and the meta-learner is trained on out-of-fold probabilities (`cv=3`), so nothing leaks from the outer test fold.

In [ ]:
def _stack_fold(name, split_id, tr, te):
    st = make_stack(name)
    st.fit(X_train.iloc[tr], y_train[tr])
    return {"model": name, "split": split_id, **all_metrics(y_train[te], st.predict(X_train.iloc[te]))}

t0 = time.time()
stack_tasks = [(n, sid, tr, te) for n in STACK_NAMES for (sid, tr, te) in outer_splits]
print(f"{len(stack_tasks)} (stack x outer-fold) tasks ...")
rows += Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(delayed(_stack_fold)(*t) for t in stack_tasks)
print(f"stacking CV finished in {time.time()-t0:.0f}s")

cv_df = pd.DataFrame(rows)
cv_summary = cv_df.groupby("model")[METRICS].agg(["mean", "std"])
cv_summary.columns = [f"{m}_{s}" for m, s in cv_summary.columns]
cv_summary = cv_summary.sort_values("severity_f1_mean", ascending=False)

pd.set_option("display.width", 220)
print("\nNESTED-CV RESULTS  (mean over outer folds, sorted by severity-F1)")
show = pd.DataFrame({m: cv_summary[f"{m}_mean"].round(4).astype(str) + " ± " + cv_summary[f"{m}_std"].round(4).astype(str)
                     for m in ["accuracy", "macro_f1", "severity_f1", "qwk", "adjacent_acc"]})
print(show.to_string())
cv_df.to_csv("final_models_cv_folds.csv", index=False)


## Step 7: Refit on the full training set and evaluate on the locked holdout

In [ ]:
def _refit_full(name, pipe, grid):
    gs = GridSearchCV(pipe, grid, cv=inner_cv, scoring=severity_scorer, n_jobs=1, refit=True, error_score="raise")
    gs.fit(X_train, y_train)
    return name, gs.best_estimator_, gs.best_params_

refit = Parallel(n_jobs=min(N_JOBS, len(MODEL_REGISTRY)), backend="loky")(
    delayed(_refit_full)(n, p, g) for n, (p, g) in MODEL_REGISTRY.items())
fitted_models = {}
for name, est, bp in refit:
    fitted_models[name] = est
    print(f"  {name:22s} best params: {bp}")
for name in STACK_NAMES:
    fitted_models[name] = make_stack(name).fit(X_train, y_train)
print("all models refit on the full training set")

hold = []
for name, est in fitted_models.items():
    hold.append({"model": name, **all_metrics(y_holdout, est.predict(X_holdout))})
holdout_df = pd.DataFrame(hold).set_index("model")
holdout_df = holdout_df.loc[cv_summary.index.intersection(holdout_df.index)]     # same order as the CV table
print("\nHOLDOUT RESULTS (locked 20%; ordered by CV severity-F1)")
print(holdout_df.round(4).to_string())
holdout_df.to_csv("final_models_holdout_results.csv")


## Step 8: Best model — classification report & confusion matrix

The winner is chosen by **CV severity-F1** (never by holdout score), so the holdout stays an unbiased final check.

In [ ]:
best_name = cv_summary.index[0]
best_est  = fitted_models[best_name]
y_hat     = best_est.predict(X_holdout)
print(f"Best by CV severity-F1: {best_name}\n")
print(classification_report(y_holdout, y_hat, target_names=CLASS_NAMES, zero_division=0, digits=3))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(confusion_matrix(y_holdout, y_hat, labels=[0, 1, 2, 3, 4]), display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title(f"Holdout confusion matrix — {best_name}")
plt.tight_layout(); plt.savefig("confusion_matrix_best.png", dpi=200); plt.show()


## Step 9: Is the difference real? Paired Wilcoxon tests on the outer folds

With `N_REPEATS = 1` there are only 5 paired scores, too few for a meaningful test (the smallest possible p-value is 0.0625). Set `N_REPEATS = 3` (15 folds) for the paper.

In [ ]:
piv = cv_df.pivot(index="split", columns="model", values="severity_f1")
n_pairs = len(piv)
best_cv = cv_summary.index[0]
if n_pairs < 10:
    print(f"Only {n_pairs} paired folds - set N_REPEATS = 3 and re-run to obtain usable p-values.")
else:
    print(f"Wilcoxon signed-rank on severity-F1, {best_cv} vs each model ({n_pairs} paired folds)")
    for other in piv.columns:
        if other == best_cv:
            continue
        try:
            p = stats.wilcoxon(piv[best_cv], piv[other]).pvalue
        except ValueError:
            p = float("nan")
        print(f"  vs {other:34s} p = {p:.4f}  ({'significant' if p < 0.05 else 'not significant'} at 0.05)")


## Step 10: Permutation importance (which clinical features matter?)

Computed on the **holdout** for the best *individual* model, using the severity-F1 scorer.

In [ ]:
indiv = [n for n in cv_summary.index if n in MODEL_REGISTRY]
imp_name = indiv[0]
pi = permutation_importance(fitted_models[imp_name], X_holdout, y_holdout, scoring=severity_scorer,
                            n_repeats=10, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=X_holdout.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(imp.index, imp.values, xerr=pd.Series(pi.importances_std, index=X_holdout.columns)[imp.index].values)
ax.set_xlabel("Drop in severity-weighted F1 when the feature is shuffled")
ax.set_title(f"Permutation importance — {imp_name}")
plt.tight_layout(); plt.savefig("permutation_importance.png", dpi=200); plt.show()
print(imp.sort_values(ascending=False).round(4).to_string())


## Notes for the paper

* **Selection protocol.** Hyper-parameters are chosen by inner CV; the reported model is chosen by CV severity-F1; the holdout is only used to report.
  Individually-tuned models and stacks are both scored on the same outer folds.
* **Stack base learners use fixed hyper-parameters** (so nothing is tuned on the outer test folds). That is conservative — a stack with tuned base learners could do slightly better — but leak-free.
* **LightGBM's starting configuration came from an Azure AutoML run over the full dataset**, which necessarily included rows that are now in the holdout. The inner grid re-selects among nearby values, but state in the paper that the seed configuration was AutoML-derived and that holdout figures for it may be slightly optimistic.
* **SMOTE on ordinal-encoded categoricals** interpolates category codes (e.g. `cp = 1.4`). This is inherited from the original pipeline for comparability; `SMOTENC` is the stricter alternative and a reasonable ablation.
* **`dataset` (collection site).** Excluded by default. Turning `USE_DATASET_SITE_FEATURE = True` raises accuracy, but part of that gain is site identity rather than clinical signal, so report it as a separate, clearly labelled experiment.
* **Class 4 (Very Severe) has ~6 holdout cases.** Recall on it is very noisy, so quote `recall_C3`/`recall_C4` with that caveat, and prefer repeated CV numbers to holdout numbers.
* **Threshold tuning** from the original notebook (classes 3 and 4) can be applied to the winner as-is: every probabilistic model here supports `predict_proba`. The ordinal model does not produce probabilities and is therefore excluded from that step.
